In [1]:
import numpy as np
import met_matching_fxnlib as lib

In [2]:
def calculate_evaporation_depth(regime_array, altitudes, RHw, T, pressures, latitude, longitude, met_type):
    # P_sat: Saturation vapor pressure (Pa)
    # RH: Relative humidity wrt ice (unitless)
    # P_atm: Atmospheric pressure (Pa)
    
    print("Initial Regime Binary:", regime_array)
    regime_array_ED = regime_array.copy()
    R = 6371*10**3 # Radius of the Earth (m)
    
    if met_type == "ERA5":
        # latitude and longitude must be floats describing the location of the GRUAN launch site
        if isinstance(latitude, float) != True:
            raise ValueError("Invalid type of", type(latitude) ,". Latitude and Longitude must be floats describing the location of the GRUAN launch site. Did you mean to pass in the GRUAN type parameter?")
        elif isinstance(longitude, float) != True:
            raise ValueError("Invalid type of", type(longitude) ,". Latitude and Longitude must be floats describing the location of the GRUAN launch site. Did you mean to pass in the GRUAN type parameter?")
        
        lat_len = 111320*0.25 # Latitude length (m). The size of a degree of latitude remains fairly constant across the Earth.
        lon_len = np.abs((2*np.pi*R*np.cos(latitude)/360)*0.25) # Longitude length (m), Haverside function
        heights = np.diff(altitudes) # Height of the gridcell (m)
        V = heights*lat_len*lon_len # volume of air in m^3
        
    elif met_type == "GRUAN":
        # latitude and longitude are altitude/pressure dependent arrays of floats describing the path of the radiosonde
        if isinstance(latitude, float) == True:
            raise ValueError("Invalid type of", type(latitude) ,". Latitude and Longitude must be arrays of floats describing the path of the GRUAN radiosonde. Did you mean to pass in the ERA5 type parameter?")
        elif isinstance(longitude, float) == True:
            raise ValueError("Invalid type of", type(longitude),". Latitude and Longitude must be arrays of floats describing the path of the GRUAN radiosonde. Did you mean to pass in the ERA5 type parameter?")
        
        lat_len = np.abs(111320*np.diff(latitude)) # Latitude length (m)
        lon_len = np.abs((2*np.pi*R*np.cos(lib.averages_between_elements(latitude))/360)*(np.diff(longitude))) # Longitude length (m), Haverside function
        height = np.diff(altitudes) # Height of the gridcell (m)
        V = height*lat_len*lon_len # volume of air in m^3
    
    else:
        raise ValueError("Invalid type. Must be 'ERA5' or 'GRUAN'.")

    contrail_molecules_water = 0 # Initialize the amount of water molecules in the volume of air
    for i in range (len(regime_array)):
        
        # If regime_array[i] == 1, the altitude is supersaturated
        if regime_array[i] == 1:
            print("Initially Supersaturated: regime_array is 1")
            # Calculate water picked up in supersaturated zone
            P_sat = lib.compute_Psat_w(T[i])
            P_atm = pressures[i]
            ppmv = (P_sat/P_atm)*(RHw[i])*10**6 # ppmv
            contrail_molecules_water = contrail_molecules_water + ppmv*V[i] # molecules of water in the volume of air

        else:
            print("Initially Subsaturated: regime_array is 0")
            # Calculate water deposited in subsaturated zone
            P_sat = lib.compute_Psat_w(T[i])
            P_atm = pressures[i]
            contrail_molecules_water_initial = contrail_molecules_water
            sat_molecules_water = (P_sat/P_atm)*10**6*V[i] # For RH = 1, molecules of water in the volume of air required for saturation
            background_molecules_water = (P_sat/P_atm)*(RHw[i])*10**6*V[i] # For RH < 1, molecules of water actually in the volume of air
            contrail_molecules_water = contrail_molecules_water - (sat_molecules_water - background_molecules_water) # Amount of water molecules deposited in the volume of air by the contrail
            print("Available water:", f"{contrail_molecules_water_initial :.2e}", 
                  "Saturation Req:", f"{sat_molecules_water - background_molecules_water:.2e}", 
                  "Remaining ppm:", f"{contrail_molecules_water:.2e}")

        # When contrail_molecules_water = 0, the contrail has evaporated
        if contrail_molecules_water > 0:
            regime_array_ED[i] = 1 # Mark subsaturated binary as supersaturated (evaporation depth)
        else:
            contrail_molecules_water = 0 # Reset the amount of water molecules in the volume of air
            print("Contrail Death, ppm reset: ", contrail_molecules_water, " ppm")
        
        print("Molecules of contrail water at altitude ", altitudes[i], "are:", f"{contrail_molecules_water:.2e}\n")
    print("Evaporation Depth Regime Binary:", regime_array_ED)
    return regime_array_ED

In [3]:
# Choose ERA5 or GRUAN to test
# GRUAN
mean_lat = 52.21
mean_lon = 14.12
std_dev = 0.25
latitude = np.random.normal(loc=mean_lat, scale=std_dev, size=6)
longitude = np.random.normal(loc=mean_lon, scale=std_dev, size=6)

# # ERA5
# latitude = 52.21 #LIN
# longitude = 14.12 #LIN

In [4]:
# Test values
regime_array = np.array([0,1,0,1,0])
regime_array_ED = regime_array # initialize
altitudes = np.array([9000,9500,10000,10500,11000,11500])
pressures = lib.alt2press(altitudes)

RHw = np.array([0.7, 1, 0.85, 1, 0.6])
T = np.array([-43.5, -46.75, -50, -53.25, 56.5, 59.75]) + 273.15

In [5]:
result = calculate_evaporation_depth(regime_array, altitudes, RHw, T, pressures, latitude, longitude, "GRUAN")

Initial Regime Binary: [0 1 0 1 0]
Initially Subsaturated: regime_array is 0
Available water: 0.00e+00 Saturation Req: 3.53e+12 Remaining ppm: -3.53e+12
Contrail Death, ppm reset:  0  ppm
Molecules of contrail water at altitude  9000 are: 0.00e+00

Initially Supersaturated: regime_array is 1
Molecules of contrail water at altitude  9500 are: 1.20e+13

Initially Subsaturated: regime_array is 0
Available water: 1.20e+13 Saturation Req: 4.06e+12 Remaining ppm: 7.89e+12
Molecules of contrail water at altitude  10000 are: 7.89e+12

Initially Supersaturated: regime_array is 1
Molecules of contrail water at altitude  10500 are: 2.04e+13

Initially Subsaturated: regime_array is 0
Available water: 2.04e+13 Saturation Req: 3.29e+16 Remaining ppm: -3.29e+16
Contrail Death, ppm reset:  0  ppm
Molecules of contrail water at altitude  11000 are: 0.00e+00

Evaporation Depth Regime Binary: [0 1 1 1 0]
